## BRONZE_LOAD INCREMENTAL
#### Incremental Bronze Ingestion with retrun-safe Watermark Logic

## Step 1 - Imports and Setup
 This Cell Imports the Pyspark and Delta helpers used in the notebook, Swithces to the correct catalog, and make sure the **Bronze Schema** exists before we start loading data. 

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable
from datetime import datetime
import uuid

In [0]:
spark.sql("USE CATALOG novacart_adb ")

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS bronze_schema")

## Step 2 - Bronze Control Table
This table stores the Watermark for each source table.
It helps the pipeline remember:
  - the latest timestamp already processed 
  - the latest primarky processed at the timestamp
  - how many rows were written in the latest run

This is what makes the Bronze load Incremental and return-safe

In [0]:
spark.sql("""
          CREATE TABLE IF NOT EXISTS novacart_adb.bronze_schema.Ingestion_Contorl(
              Layer STRING,
              Table_Name STRING,
              Ts_Col  STRING,
              Pk_Col  STRING,
              Last_Successful_Ts TIMESTAMP,
              Last_Successful_Pk STRING,
              Last_Run_Id STRING,
              Rows_Written BIGINT,
              Run_Status STRING,
              Updated_At TIMESTAMP
          )
          USING DELTA
          """)

## Step 3 - Source Table Configuration
This cell define which source tables will be loaded into Bronze and which columns should be used as 
  - **Primary Key**
  - **Timestamp/Watermark Column**

In [0]:
Table_Config = {
    "orders"   : {"Pk_Col" : "order_id", "Ts_Col" : "Updated_at"},
    "Products" : {"Pk_Col" : "product_id", "Ts_Col" : "Updated_at"},
    "payments" : {"Pk_Col" : "payment_id", "Ts_Col" : "Processed_at"}
}

bronze_run_id = str(uuid.uuid4())
print("Current Bronze Run ID:",bronze_run_id)

## Step 4 Helper Functions
This Cell contains reusable functions:
  - get_last_successful_watermark()
  - upsert_bronze_control()

This functions keep main load logic cleaner and easier to undestand.


In [0]:
def get_last_successful_watermark(table_name: str):
    ctrl = (
        spark.table("novacart_adb.bronze_schema.ingestion_contorl")
        .filter(
            (F.col("layer") == "bronze")
            & (F.col("Table_Name") == table_name)
            & (F.col("Run_Status") == "Success")
        )
        .orderBy(F.col("Updated_At").desc())
        .limit(1)
    )

    rows = ctrl.collect()
    if not rows:
        return None, None

    return rows[0]["Last_Successful_Ts"], rows[0]["Last_Successful_Pk"]

In [0]:
def upsert_bronze_control(
    table_name, ts_col, pk_col, last_ts, last_pk, rows_written, run_id
):
    control_df = spark.createDataFrame(
        [
            (
                "bronze",
                table_name,
                ts_col,
                pk_col,
                last_ts,
                int(last_pk) if last_pk is not None else None,
                run_id,
                int(rows_written),
                "Success",
                datetime.utcnow(),
            )
        ],
        schema="""
            Layer STRING,
            Table_Name STRING,
            Ts_Col  STRING,
            Pk_Col  STRING,
            Last_Successful_Ts TIMESTAMP,
            Last_Successful_Pk STRING,
            Last_Run_Id STRING,
            Rows_Written BIGINT,
            Run_Status STRING,
            Updated_At TIMESTAMP
        """,
    )
    dt = DeltaTable.forName(spark, "novacart_adb.bronze_schema.ingestion_contorl")
    (
        dt.alias("T")
        .merge(
            control_df.alias("S"), "T.Table_Name == S.Table_Name and T.layer == S.layer"
        )
        .whenMatchedUpdate(
            set={
                "ts_col": "S.ts_col",
                "pk_col": "S.Pk_col",
                "last_successful_ts": "S.Last_Successful_Ts",
                "last_successful_pk": "S.Last_Successful_Pk",
                "last_run_id": "S.Last_Run_Id",
                "rows_written": "S.Rows_Written",
                "run_status": "S.Run_Status",
                "updated_at": "S.Updated_At",
            }
        )
        .whenNotMatchedInsertAll()
        .execute()
    )

## Step 5 - Bronze Incremental Load loop
This is Main bronze logic.

For each table the notebook:

  - Reads the last watermark.
  - reads the Source SQL table.
  - Filters Only new/ Changed Records
  - adds Bronz audit Column.
  - append rown into the bronze DeltaTable.
  - Update the control table
  
This is the core incremental loading logic


In [0]:
for table_name, cfg in Table_Config.items():

    pk_col = cfg["Pk_Col"]
    ts_col = cfg["Ts_Col"]

    source_table = f"novacart_sql_connection_catalog.novacart.{table_name}"
    target_table = f"novacart_adb.bronze_schema.{table_name}_raw"

    last_successful_ts, last_successful_pk = get_last_successful_watermark(table_name)

    print(f"=== Processing {table_name} ===")
    print(f"Last_successful_ts: {last_successful_ts}")
    print(f"Last_successful_pk: {last_successful_pk}")

    # Read source
    source_df = spark.read.table(source_table).withColumn(
        ts_col, F.col(ts_col).cast("timestamp")
    )

    # Incremental filter (RETURN-SAFE)
    if last_successful_ts is None:
        rows_to_load = source_df
    else:
        rows_to_load = source_df.filter(
            (F.col(ts_col) > F.lit(last_successful_ts))
            | (
                (F.col(ts_col) == F.lit(last_successful_ts))
                & (F.col(pk_col).cast("long") > F.lit(last_successful_pk))
            )
        )

    # Add audit columns
    rows_to_load = (
        rows_to_load.withColumn("bronze_ingested_at", F.current_timestamp())
        .withColumn("bronze_run_id", F.lit(bronze_run_id))
        .withColumn("bronze_source_table", F.lit(source_table))
    )

    rows_count = rows_to_load.count()
    print(f"{table_name} rows_to_load = {rows_count}")

    if rows_count == 0:
        print(f"No new rows for {table_name}")

        upsert_bronze_control(
            table_name,
            ts_col,
            pk_col,
            last_successful_ts,
            last_successful_pk,
            rows_count,
            bronze_run_id,
        )
        continue

    # Write to bronze
    rows_to_load.write.format("delta").mode("append").saveAsTable(target_table)

    # Compute new watermark
    max_ts = rows_to_load.agg(F.max(ts_col).alias("max_ts")).collect()[0]["max_ts"]

    max_pk = (
        rows_to_load.filter(F.col(ts_col) == F.lit(max_ts))
        .agg(F.max(F.col(pk_col).cast("long")).alias("max_pk"))
        .collect()[0]["max_pk"]
    )

    # Update control table
    upsert_bronze_control(
        table_name, ts_col, pk_col, max_ts, max_pk, rows_count, bronze_run_id
    )

    print(f"Wrote {rows_count} rows to {target_table}")

In [0]:
%sql
select * from novacart_adb.bronze_schema.ingestion_contorl

In [0]:
%sql
SELECT * FROM novacart_adb.bronze_schema.orders_raw